In [1]:
import numpy as np

def run_market_clearing_admm(d, rho=1.0, max_iter=50):
    """
    Solves the Exchange Problem using ADMM.
    
    Parameters:
    d   : (N,) array of preferred trading amounts (demand/supply targets)
    rho : Penalty parameter (step size for convergence)
    """
    N = len(d)
    
    # Initialize variables
    x = np.zeros(N) # Agent's actual trade
    z = np.zeros(N) # Market validated trade
    y = np.zeros(N) # Price/Dual variable (per agent)
    
    print(f"--- Starting Market Clearing (N={N} agents) ---")
    print(f"Initial Net Imbalance: {np.sum(d):.4f} units")
    
    history = []

    for k in range(max_iter):
        
        # --- 1. x-update (Decentralized Agent Step) ---
        # Formula: x = (d + rho*z - y) / (1 + rho)
        # Each agent calculates this locally without knowing other d_j
        x = (d + rho * z - y) / (1 + rho)
        
        # --- 2. z-update (Centralized Clearing Step) ---
        # Project (x + y/rho) onto the set where sum(z) = 0
        # Formula: z = v - mean(v)
        v = x + y / rho
        z = v - np.mean(v)
        
        # --- 3. Dual update (Price Step) ---
        # Formula: y = y + rho * (x - z)
        resid = x - z
        y = y + rho * (x - z)
        
        # --- Diagnostics ---
        # The market imbalance is the sum of x
        market_imbalance = np.abs(np.sum(x))
        # The primal residual is ||x - z||
        primal_resid = np.linalg.norm(resid)
        
        history.append(market_imbalance)
        
        if k % 10 == 0:
            print(f"Iter {k}: Market Imbalance = {market_imbalance:.6f}, Primal Resid = {primal_resid:.6f}")

    return x, z, history

# ==========================================
# SIMULATION
# ==========================================
if __name__ == "__main__":
    np.random.seed(42)
    
    # 1. Setup Scenario
    N_agents = 50
    # d represents desired trade. 
    # Positive = Need power (Buyer). Negative = Have solar (Seller).
    # We create a random market that is NOT naturally balanced initially.
    d = np.random.uniform(-10, 10, N_agents)
    
    # 2. Solve
    rho_val = 1.0
    x_final, z_final, imbalance_hist = run_market_clearing_admm(d, rho=rho_val)
    
    # 3. Analysis
    print("\n--- Final Results ---")
    print(f"Original Net Demand (Sum d): {np.sum(d):.4f}")
    print(f"Final Net Trade (Sum x):     {np.sum(x_final):.4f} (Should be 0)")
    
    print("\nExample Agents (First 5):")
    print(f"{'Agent':<6} {'Target (d)':<12} {'Final Trade (x)':<15} {'Adjustment':<12}")
    for i in range(5):
        adj = x_final[i] - d[i]
        print(f"{i:<6} {d[i]:<12.4f} {x_final[i]:<15.4f} {adj:<12.4f}")
        
    print("\nInterpretation:")
    avg_adj = np.mean(x_final - d)
    if avg_adj < 0:
        print(f"The market had excess demand. Everyone had to reduce consumption by avg {abs(avg_adj):.4f}.")
    else:
        print(f"The market had excess supply. Everyone had to increase consumption by avg {avg_adj:.4f}.")

--- Starting Market Clearing (N=50 agents) ---
Initial Net Imbalance: -54.0761 units
Iter 0: Market Imbalance = 27.038048, Primal Resid = 3.823757
Iter 10: Market Imbalance = 0.026404, Primal Resid = 0.003734
Iter 20: Market Imbalance = 0.000026, Primal Resid = 0.000004
Iter 30: Market Imbalance = 0.000000, Primal Resid = 0.000000
Iter 40: Market Imbalance = 0.000000, Primal Resid = 0.000000

--- Final Results ---
Original Net Demand (Sum d): -54.0761
Final Net Trade (Sum x):     -0.0000 (Should be 0)

Example Agents (First 5):
Agent  Target (d)   Final Trade (x) Adjustment  
0      -2.5092      -1.4277         1.0815      
1      9.0143       10.0958         1.0815      
2      4.6399       5.7214          1.0815      
3      1.9732       3.0547          1.0815      
4      -6.8796      -5.7981         1.0815      

Interpretation:
The market had excess supply. Everyone had to increase consumption by avg 1.0815.
